# Lesson 17.4: Make the operating contract inspectable

Read the matching lesson at `modules/17-durable-volume/04-operating-contract.md`. Run this notebook from top to bottom in the project `.venv` kernel. All cases are fictional. The default path sends no model-provider requests. A kernel retains values between cells; restart and run all after an experiment to check independence.

## The practical question

Preparing more cases per minute can overload the people who accept or investigate them. Harbour therefore examines technical concurrency and human review capacity together, while preserving ownership when workers fail or become slow.

In [ ]:
from ai_orgdesign.labs import m17 as lab
print("Module 17 code loaded; no provider called.")

## Follow the mechanism

bounded_work uses an asyncio semaphore to limit the number of active coroutines. Each coroutine enters the semaphore, increments a counter, awaits a short artificial delay and decrements the counter in finally. gather collects results in input order. The separate SQLite job exercise gives a worker a time-limited lease through one conditional UPDATE. Another worker may reclaim an expired lease, but finish accepts only the current owner with an unexpired lease. The supplied clock values make expiration tests deterministic.

In [ ]:
import inspect
print(inspect.getsource(lab.bounded_work))

## Predict and run

At concurrency two, the peak counter never exceeds two. The first worker can claim job-1 at time zero; a second cannot claim it at time five, but can reclaim it at eleven after a ten-second lease. A former owner cannot finish after the transfer. These are local ownership tests, not a benchmark of distributed throughput.

In [ ]:
from ai_orgdesign.labs.m17 import bounded_work, demo
result = await bounded_work(list(range(6)), limit=2)
assert result["peak"] <= 2
print(result)
print(demo())

## Exercise

Assume thirty cases per hour enter review and reviewers can finish twenty per hour. What changes if agent preparation becomes twice as fast? How should a lease handle an external action completed just before a worker fails?

Pause and write the predicted result and its interpretation before opening the worked answer below.

## Worked answer

The review backlog grows by ten cases each hour under the stated steady rates. Faster preparation does not increase review capacity and may expose the bottleneck sooner. A lease alone cannot establish whether an external action happened. Use a stable command identity, the external service's idempotency facilities where available, and reconciliation before retrying an uncertain action.

In [ ]:
from ai_orgdesign.labs.m17 import bounded_work
serial = await bounded_work([1,2,3], limit=1)
assert serial["peak"] == 1
print(serial)

## Alternatives and organisational meaning

A serial queue is easier to reason about and may be adequate at low volume. More concurrency can hide waiting but adds contention and ordering problems. Distributed brokers and worker pools introduce acknowledgements, redelivery and operational responsibilities that an in-process semaphore does not supply.

Explain separately what executed, what its output establishes and what remains a question for the surrounding service.